# Exercise: Before development

Welcome to the first notebook of the advanced topic module. 

In this notebook, we will practice the data management practices seen in the video. We will use an air quality dataset to exemplify some of the explained practices. This dataset is part of the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/360/air+quality).

## Libraries

For running this notebook, we will need the following libraries

In [ ]:
from ucimlrepo import fetch_ucirepo
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

## Understand the data


Air quality is a crucial aspect of environmental health, directly impacting the well-being of both ecosystems and human populations. Among the various pollutants, nitrogen dioxide ($NO_2$) is of particular concern due to its harmful effects on respiratory health and the environment. $NO_2$ is a byproduct of combustion processes, primarily from vehicle emissions and industrial activities.

"The science on and knowledge about health effects of $NO_2$ air pollution has increased significantly over the last ten to fifteen years. It is now confirmed that $NO_2$ exposure can lead to respiratory and circulatory premature death (from both short- and long-term exposure), to the development of asthma in children and in adults, to bronchitis in children, and it can deteriorate the health of people living with respiratory or cardiovascular disease." [(HEAL, 2023)](https://www.env-health.org/science-review-the-health-impacts-of-nitrogen-dioxide-no2/#:~:text=It%20is%20now%20confirmed%20that%20NO2%20exposure%20can,of%20people%20living%20with%20respiratory%20or%20cardiovascular%20disease.)

<center><div>
<img src="https://images.unsplash.com/photo-1607432750402-48f85c94f63a?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D" width="700"/>
</div></center>


Accurate measurement and monitoring of NO2 levels are crucial for several reasons. Reliable data on NO2 concentrations is essential for understanding the extent of air pollution and its potential health impacts. 

In the context of developing a machine learning application, some researchers have developed a new method to measure multiple air pollutants simultaneously, including $NO_2$. This method is based on the use of a low-cost gas multisensor, which is a promising alternative to the current reference method. However, in order to use it, the researchers need a calibration function that relates the measured values of their method to the values given by the reference method. We will develop that calibration function using machine learning. Nevertheless, before we can start developing our machine learning model, we need to understand the data we will be working with.

### Description of the Dataset

To calibrate their multisensor, the researchers recorded the concentrations of different pollutants using two different instruments: their gas multisensor and a certified analyzer (as reference method). The measurement campaign took place at a main street located in the centre of an Italian city characterized by heavy car traffics.

The researchers recorded the hourly average concentrations of different pollutants using both devices for one year (March 2004 to February 2005). The dataset contains 9357 instances of hourly averaged responses.

Let's load the dataset and take a look at the first few rows.

In [ ]:
# fetch dataset 
air_quality = fetch_ucirepo(id=360) 
  
# data (as pandas dataframes) 
raw_dataset = air_quality.data.features

In [ ]:
raw_dataset

We can have an idea about the shape of the dataset we are dealing with. All the $observations/measurements$ are the $rows$ and all the $features$ represent the $columns$ of the dataframe.

In [ ]:
samples, features = raw_dataset.shape
print(f'Number of observations/rows are {samples}')
print(f'Number of features/columns are {features}')

In [ ]:
raw_dataset

### Insights to the features of the dataset

Before we start any sort of modelling or analysis, we need to understand what each of the different features in the dataset means.

The dataset has 15 features. For every pollutant there are two types of measurement value: $measured$ and $targeted$. The $measured$ value is the pollutant concentration reported by the experimental multisensor whereas the $targeted$ concentration refers to the value given by the certified analyzer. For each observation, the researchers registered the date and hour, the temperature, relative humidity, and absolute humidity.

Further descriptions of every pollutant value is elaborated below:

*   **Hourly concentration CO in mg/m^3 ($targeted \_ CO$)**: Hourly averaged concentration of Carbon Monoxide (CO) measured in milligrams per cubic meter.
*  **PT08.S1 (tin oxide) hourly averaged sensor response ($measured\_ CO$)**: It is the Hourly averaged sensor response of the first metal oxide sensor (tin oxide) targeted for Carbon Monoxide (CO) detection.

* **Non Metanic HydroCarbons (NMHC) concentration in microg/m³($targeted \_ NMHC$)**: Concentration of Non Metanic HydroCarbons (NMHC) measured in micrograms per cubic meter using a reference analyzer.
* **PT08.S2 (titania) hourly averaged sensor response ($measured \_ NMHC$)**: Hourly averaged sensor response of the second metal oxide sensor (titania) targeted for Non Metanic HydroCarbons (NMHC) detection.
* **Benzene concentration in microg/m³ ($targeted \_ Benzene$):** Concentration of Benzene measured in micrograms per cubic meter.

* **True hourly averaged NOx concentration in ppb ($targeted \_ NO_x$)**: True hourly averaged concentration of Nitrogen Oxides ($NO_x$) measured in parts per billion using a reference analyzer.
* **PT08.S3 (tungsten oxide) hourly averaged sensor response ($measured \_ NO_x$)**: Hourly averaged sensor response of the third metal oxide sensor (tungsten oxide) targeted for Nitrogen Oxides ($NO_x$) detection.

* **True hourly averaged NO2 concentration in microg/m³($targeted \_ NO_2$):** True hourly averaged concentration of Nitrogen Dioxide (NO2) measured in micrograms per cubic meter using a reference analyzer.
* **PT08.S4 (tungsten oxide) hourly averaged sensor response ($measured \_ NO_2$)**: Hourly averaged sensor response of the fourth metal oxide sensor (tungsten oxide) targeted for Nitrogen Dioxide ($NO_2$) detection.

* **PT08.S5 (indium oxide) hourly averaged sensor response ($measured \_ O_3$)**: Hourly averaged sensor response of the fifth metal oxide sensor (indium oxide) targeted for Ozone ($O_3$) detection.

**Note**: The description of the dataset also warns us about the presence of missing values in the dataset. They are represented by -200 value. We will deal with them later in the notebook.

## Data splitting

Before continuing with further analysis of the data, we are going to separate the data into three subsets: training, validation, and testing.

**Training Data:**  This set of data is used for learning by the model, that is, to fit the parameters of the machine learning model.

**Validation Data:** Set of data used to provide an unbiased evaluation of a model fitted on the training dataset while tuning model hyperparameters.

**Test Data:** Set of data used to provide an unbiased evaluation of a final model fitted on the training dataset.

We will assume that each measurement is independent of the others. Therefore, we can randomly split the data into the three subsets. We will use 70% of the data for training, 20% for validation, and 10% for testing.

In [ ]:
# Shuffling the rows
shuffled_dataset = raw_dataset.sample(frac = 1, random_state = 42)

In [ ]:
shuffled_dataset.head()

**Exercise 1:**

Split the dataset into training, validation, and testing subsets. Use the following names for the variables: `training_data`, `validation_data`, `test_data`. Use the following proportions: 70% for training, 20% for validation, and 10% for testing.

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

## Data cleaning

After this point, we will develop our cleaning process in the training data and then apply the same process to the validation and test data. This is to avoid any data leakage from the validation and test data into the training data.

### Data renaming

The first step in this case would be to change the name of the columns to make them more readable.

In [ ]:
def rename_columns(dataset):
    # Renaming 'NO2(GT)' to 'Targeted NO2' and 'PT08.S4(NO2)' to 'Measured NO2'
    renamed_dataset = dataset.rename(columns = {'NO2(GT)':'targeted_NO2', 
                                                'PT08.S4(NO2)':'measured_NO2', 
                                                'CO(GT)': 'targeted_CO',
                                                'PT08.S1(CO)': 'measured_CO',
                                                'NMHC(GT)': 'targeted_NMHC',
                                                'PT08.S2(NMHC)': 'measured_NMHC',
                                                'C6H6(GT)': 'targeted_C6H6',
                                                'NOx(GT)': 'targeted_NOx',
                                                'PT08.S3(NOx)': 'measured_NOx',
                                                'PT08.S5(O3)': 'measured_O3',
                                                })
    return renamed_dataset

In [ ]:
training_data = rename_columns(training_data)

In [ ]:
print(training_data.columns)

### Data checking

In this moment, we can have some sanity checks to catch possible errors in our dataset. Some of the checks we can do are:
1. Test that our values are positive
2. Test that the relative humidity values are between 0 and 100
3. Test that the temperature values are between -30 and 50 (Celsius)
4. ...

**Exercise 2:**

Perform the sanity checks mentioned above. Use the assert function to check that the values are correct. If the values are not correct, raise an error with a message explaining the problem.

For the fourth check, you can come up with your own sanity check.

In [ ]:
def quality_check(dataset):
    try:
        # ---------------------- student exercise --------------------------------- #
        # YOUR CODE HERE
        # ---------------------- student exercise --------------------------------- #
        
    except AssertionError as e:
        print('The checks failed!: ', e.args[0])
    else:
        print('The dataset passed the checks!')


In [ ]:
quality_check(training_data)

### Checking for missing values

We see that this code generates errors. This means that there are some values that do not meet the conditions we have imposed. In this case, the errors are due to the presence of missing values. Those missing values were set to the value of -200. We will replace those values with NaN values.

**Exercise 3:**

Replace the -200 values with NaN values. Use the `replace` function from pandas.

In [ ]:
def replace_missing_values(dataset):
    # Replacing -200 values with NaN
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
    return dataset

training_data = replace_missing_values(training_data)

Let's count how many missing values we have in each column and what percentage they represent.

In [ ]:
na_counts = training_data.isna().sum()
na_proportions = round(na_counts / len(training_data), 2)
na_data = pd.concat([na_counts, na_proportions], axis=1, keys=['NA Count', 'NA Proportion'])
na_data

In [ ]:
quality_check(training_data)

### Feature selection

Now, we will select the features we will use for our model. We will use the following features:
- $measured\_ CO$
- $measured \_ NMHC$
- $measured \_ NO_x$
- $measured \_ NO_2$
- $measured \_ O_3$
- $Temperature$
- $Relative Humidity$
- $Absolute Humidity$

We chose these features because they are the ones that are measured by the multisensor. In this way, we will be able to use the multisensor to predict the concentration of NO2.

And we will predict the following features:
- $targeted \_ NO_2$

**Exercise 4:**

Return a dataframe with the selected features.

In [ ]:
def select_features(dataset):
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
    return subset_data

In [ ]:
selected_dataset = select_features(training_data)

After all the cleaning, let's see the statistics of the dataset.

In [ ]:
selected_dataset.describe()

### Removing nan values

In [ ]:
def remove_nan_values(dataset):
    return dataset.dropna()

In [ ]:
selected_dataset = remove_nan_values(selected_dataset)

In [ ]:
selected_dataset.describe()

Notice that the count value was reduced from 6297 to 5185.

## Data visualization

The aim of data visualization is to present complex data in a visual format that is easy to understand, interpret, and derive insights from. Visualization helps in identifying patterns, trends, correlations, and outliers within the data.

In this exercise the plots of $targeted \_ NO2$ will be done against the explanatory variables.

Converting all the columns of the dataframe into NumPy. This method is illustrated below

In [ ]:
# Extracting every feature and converting it into numpy
T = selected_dataset['T'].to_numpy()
RH = selected_dataset['RH'].to_numpy()
AH =selected_dataset['AH'].to_numpy()
measured_CO = selected_dataset['measured_CO'].to_numpy()
measured_NMHC = selected_dataset['measured_NMHC'].to_numpy()
measured_NOx = selected_dataset['measured_NOx'].to_numpy()
measured_NO2 = selected_dataset['measured_NO2'].to_numpy()
measured_O3 = selected_dataset['measured_O3'].to_numpy()

targeted_NO2 = selected_dataset['targeted_NO2'].to_numpy()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 10))  # rows = 2, cols= 4

opacity = 0.2

# Plot TargetNO2 vs. Measured NO2
axs[0][0].scatter(measured_NO2, targeted_NO2, color = 'orange', alpha=opacity)
axs[0][0].set_xlabel('Measured $NO_2$')
axs[0][0].set_ylabel('Target $NO_2$')
axs[0][0].set_title('Measured $NO_2$ effect on Target $NO_2$')

# Plot Targeted NO2 vs. measured_CO
axs[0][1].scatter(measured_CO, targeted_NO2, color = 'magenta', alpha=opacity)
axs[0][1].set_xlabel('Measured CO')
axs[0][1].set_ylabel('Target $NO_2$')
axs[0][1].set_title('Measured CO effect on Target $NO_2$')

# Plot Targeted NO2 vs. measured_NMHC
axs[0][2].scatter(measured_NMHC, targeted_NO2, color = 'gray', alpha=opacity)
axs[0][2].set_xlabel('Measured NMHC')
axs[0][2].set_ylabel('Target $NO_2$')
axs[0][2].set_title('Measured NMHC effect on Target $NO_2$')

# Plot Targeted NO2 vs. measured_NOx
axs[0][3].scatter(measured_NOx, targeted_NO2, color = 'purple', alpha=opacity)
axs[0][3].set_xlabel('Measured NOx')
axs[0][3].set_ylabel('Target $NO_2$')
axs[0][3].set_title('Measured NOx effect on Target $NO_2$')

# Plot Targeted NO2 vs. measured_O3
axs[1][0].scatter(measured_O3, targeted_NO2, color = 'black', alpha=opacity)
axs[1][0].set_xlabel('Measured O3')
axs[1][0].set_ylabel('Target $NO_2$')
axs[1][0].set_title('Measured O3 effect on Target $NO_2$')

# Plot Targeted NO2 vs. Temperature
axs[1][1].scatter(T, targeted_NO2, color = 'red', alpha=opacity)
axs[1][1].set_xlabel('Temperature')
axs[1][1].set_ylabel('Target $NO_2$')
axs[1][1].set_title('Temperature effect on Target $NO_2$')

# Plot Targeted NO2 vs. RH
axs[1][2].scatter(RH, targeted_NO2, color = 'cyan', alpha=opacity)
axs[1][2].set_xlabel('Relative Humidity (%RH)')
axs[1][2].set_ylabel('Target $NO_2$')
axs[1][2].set_title('Relative Humidity effect on Target $NO_2$')

# Plot Targeted NO2 vs. AH
axs[1][3].scatter(AH, targeted_NO2, color = 'green', alpha=opacity)
axs[1][3].set_xlabel('Absolute Humidity (AH)')
axs[1][3].set_ylabel('Target $NO_2$')
axs[1][3].set_title('Absolute Humidity effect on Target $NO_2$')

plt.tight_layout()
plt.show()

From the above 2D-plots you can definitely see that there is a correlation of $targeted \_NO2$ with other parameters. So, even though each sensor was especialized for a particular gas, they can be useful to estimate the concentration of other pollutants. 

To reinforce what we observed in the plots, we can compute the correlation coefficient of $targeted \_NO2$ with respect to the other parameters.

In [ ]:
correlation = selected_dataset.corr()['targeted_NO2']

In [ ]:
print(correlation)

In [ ]:
plt.hist(selected_dataset['targeted_NO2'], bins=20, color='blue', edgecolor='black')
plt.xlabel('Targeted NO2')
plt.ylabel('Frequency')
plt.title('Histogram of Targeted NO2')
plt.tight_layout()
plt.show()


Now the dataset is cleaned and split into training, validation and testing. We now scale the data

In [ ]:
def separate_features_and_target(dataset):
    X = dataset.drop(columns=['targeted_NO2'])
    y = dataset['targeted_NO2']
    return X, y

X_train, y_train = separate_features_and_target(selected_dataset)

**Exercise 5:**

Scale the data using the `StandardScaler` from `sklearn.preprocessing`. Use the `fit_transform` function to scale the training data. Notice that for the y_train variable, we need to use the `values` and `reshape` functions to extract the values and reshape the array into a column vector.

In [ ]:
#Scaling X_train
X_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fit on training data and transform it
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

We have scaled the values, now its important to know the parameters that were used to scaling the values i.e. the *Mean* and the *Standard Deviation* as these are the values that will be used to 'translate' the scaled values back to the original values.

In [ ]:
plt.hist(y_train_scaled, bins=20, color='blue', edgecolor='black')
plt.xlabel('Scaled Targeted NO2')
plt.ylabel('Frequency')
plt.title('Histogram of Scaled Targeted NO2')
plt.tight_layout()
plt.show()

## Data exporting

In this notebook, we explored the data and created functions to convert it from the raw format to the format we will use in our machine learning model. We can apply those functions to the validation and test data and save the data in a format that can be used in the next notebooks.

In [ ]:
validation_data = rename_columns(validation_data)
validation_data = replace_missing_values(validation_data)

quality_check(validation_data)

validation_data = select_features(validation_data)
validation_data = remove_nan_values(validation_data)
X_val, y_val = separate_features_and_target(validation_data)

X_val_scaled = X_scaler.transform(X_val)
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1))

**Exercise 6:**

Repeat the process with the test set. 

In [ ]:
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

In [ ]:
pd.DataFrame(X_train_scaled, columns=X_train.columns).to_csv("Scaled_Training_features.csv", index = False)
pd.DataFrame(y_train_scaled, columns=['targeted_NO2']).to_csv("Scaled_Training_targets.csv", index = False)

pd.DataFrame(X_val_scaled, columns=X_val.columns).to_csv("Scaled_Validation_features.csv", index = False)
pd.DataFrame(y_val_scaled, columns=['targeted_NO2']).to_csv("Scaled_Validation_targets.csv", index = False)

pd.DataFrame(X_test_scaled, columns=X_test.columns).to_csv("Scaled_Testing_features.csv", index = False)
pd.DataFrame(y_test_scaled, columns=['targeted_NO2']).to_csv("Scaled_Testing_targets.csv", index = False)

Furthermore, we need to save the scalers to normalize and unnormalize our data. We will save them in a pickle file.

In [ ]:
# Export X_scaler
with open('X_scaler.pk', 'wb') as f:
    pickle.dump(X_scaler, f)

# Export y_scaler
with open('y_scaler.pk', 'wb') as f:
    pickle.dump(y_scaler, f)


## Next Steps

In this notebook we cleaned, scaled and visualized the data, in the next notebook we shall use this cleaned data to run Machine Learning experiments.